# SIDD Orthorectification — Pre-Projected Product Re-Orthorectification

**Objective**: Orthorectify a SIDD (SAR Image Data Display) product, which is already in a projected geometry, to a custom output grid for analysis or overlay.

## Overview

**SIDD vs SICD**:
- **SICD**: Complex SAR image in **slant-range geometry** (sensor-native)
- **SIDD**: Detected magnitude in **pre-projected geometry** (ground plane or slant plane)

SIDD products are already **orthorectified** by the producer (using their DEM and projection choice). This notebook demonstrates:
1. **Re-orthorectifying** SIDD to a different projection (e.g., UTM zone, custom pixel size)
2. Understanding **SIDD geolocation** (affine-like vs SICD R/Rdot)
3. When to use SIDD vs SICD for ortho workflows

## Theory

### SIDD Geometry

SIDD metadata includes:
- **PlaneProjection**: `SlantPlane` or `GroundPlane`
- **ProductPlane**: Reference surface (ellipsoid or HAE plane)
- **SampleSpacing**: Row/col pixel spacing in meters
- **ReferencePoint**: Origin (0,0) geo-coordinates

SIDD geolocation is **simpler** than SICD:
- SICD: Iterative R/Rdot inverse (nonlinear, exact)
- SIDD: Plane projection + affine-like transform (linear, approximate)

### When to Re-Orthorectify SIDD

Use cases:
- **Different projection**: SIDD in one UTM zone, analysis requires another
- **Finer resolution**: Original SIDD at 1m, need 0.5m for detail
- **Co-registration**: Align SIDD with other layers (EO, MSI)
- **Custom DEM**: Producer used SRTM, you have higher-res FABDEM

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | SIDD reader via factory (`get_reader('sidd', ...)`) |
| `grdl.geolocation` | `create_geolocation()` auto-detect SIDD geolocation |
| `grdl.image_processing.ortho` | `orthorectify()`, `UTMGrid` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path
import gc
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.image_processing.ortho import orthorectify, UTMGrid

## Configuration — User Paths

In [ ]:
# Path to SIDD file
SIDD_PATH = Path("/path/to/your/sidd_file.nitf")

# Validate path
assert SIDD_PATH.exists(), f"SIDD file not found: {SIDD_PATH}"

print(f"✓ SIDD file: {SIDD_PATH.name}")

## Step 1: Load SIDD Metadata and Extract Chip

In [ ]:
with get_reader('sidd', SIDD_PATH) as reader:
    meta = reader.metadata
    rows = meta.measurement.pixel_footprint.row
    cols = meta.measurement.pixel_footprint.col
    
    print(f"Image size: {rows} × {cols}")
    print(f"Product type: {meta.product_creation.product_type}")
    
    # Display SIDD-specific geometry
    if meta.measurement.projection_ref_point:
        ref = meta.measurement.projection_ref_point
        print(f"Reference point: ({ref.row_col.row:.1f}, {ref.row_col.col:.1f})")
        print(f"  → ({ref.ecf.x:.1f}, {ref.ecf.y:.1f}, {ref.ecf.z:.1f}) ECEF (m)")
    
    if meta.measurement.plane_projection:
        print(f"Plane projection: {meta.measurement.plane_projection.projection_type}")
        if meta.measurement.plane_projection.sample_spacing:
            sp = meta.measurement.plane_projection.sample_spacing
            print(f"Sample spacing: row={sp.row:.3f} m, col={sp.col:.3f} m")
    
    # Extract center chip
    chip_size = min(4096, rows, cols)
    r_start = (rows - chip_size) // 2
    c_start = (cols - chip_size) // 2
    
    chip = reader.read(
        row_start=r_start, row_end=r_start + chip_size,
        col_start=c_start, col_end=c_start + chip_size,
    )

print(f"\nExtracted chip: {chip.shape}, dtype: {chip.dtype}")

## Step 2: Create SIDD Geolocation

In [ ]:
from grdl.geolocation.chip import ChipGeolocation

with get_reader('sidd', SIDD_PATH) as reader:
    # Auto-detect SIDD geolocation
    geo_full = create_geolocation(reader)
    
    # Wrap with chip offset
    geo = ChipGeolocation(
        geolocation=geo_full,
        row_offset=r_start,
        col_offset=c_start,
    )

print(f"✓ Geolocation: {type(geo_full).__name__}")
print(f"  Chip offset: row={r_start}, col={c_start}")

## Step 3: Define Output Grid — UTM

In [ ]:
with get_reader('sidd', SIDD_PATH) as reader:
    meta = reader.metadata
    # SIDD reference point
    if meta.geo_data and meta.geo_data.scp:
        scp_lat = meta.geo_data.scp.llh.lat
        scp_lon = meta.geo_data.scp.llh.lon
    else:
        # Fallback: use projection ref point
        ref = meta.measurement.projection_ref_point
        # Convert ECEF to geodetic
        from grdl.geolocation.coordinates import ecef_to_geodetic
        scp_lat, scp_lon, _ = ecef_to_geodetic(ref.ecf.x, ref.ecf.y, ref.ecf.z)

# UTM grid centered on SCP
output_grid = UTMGrid.from_center(
    center_lat=scp_lat,
    center_lon=scp_lon,
    width_m=chip_size * 0.5,
    height_m=chip_size * 0.5,
    pixel_size_m=0.5,
)

print(f"UTM Grid:")
print(f"  Zone: {output_grid.utm_zone}{output_grid.hemisphere}")
print(f"  Size: {output_grid.shape}")
print(f"  Pixel size: {output_grid.pixel_size_m} m")

## Step 4: Orthorectify SIDD to Custom Grid

In [ ]:
print("Running SIDD orthorectification...")

ortho_result = orthorectify(
    data=chip,
    geolocation=geo,
    output_grid=output_grid,
    interpolation='bilinear',
)

ortho_data = ortho_result.data

print(f"✓ Orthorectified: {ortho_data.shape}")
print(f"  Valid pixels: {np.sum(~np.isnan(ortho_data))} / {ortho_data.size}")

## Step 5: Visualization — Original vs Re-Ortho

In [ ]:
viewer = napari.Viewer(title="SIDD Orthorectification")

# Layer 1: Original SIDD (already projected by producer)
viewer.add_image(
    chip,
    name="Original SIDD",
    colormap="gray",
)

# Layer 2: Re-orthorectified to custom UTM grid
viewer.add_image(
    ortho_data,
    name="Re-Ortho UTM",
    colormap="gray",
    visible=True,
)

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print("✓ Napari viewer opened")
print("  Layer 1: Original SIDD projection (producer's)")
print("  Layer 2: Re-orthorectified to custom UTM grid")
print("\nExecution paused — close the napari window to continue and free memory...")
napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del chip, ortho_data, geo, viewer
gc.collect()
print("✓ Memory cleanup complete")

## Physical Interpretation

### SIDD Pre-Projection

SIDD products are **already orthorectified** by the data provider:
- Magnitude detected (no complex values)
- Projected to ground plane or slant plane
- Often includes contrast enhancement (Mangis Density, PEDF)

**Advantage**: Ready for display, no processing needed  
**Limitation**: Fixed projection, fixed DEM (if terrain-corrected)

### Re-Orthorectification Use Cases

1. **Projection mismatch**: SIDD in UTM Zone 17N, analysis requires Zone 18N
2. **Resolution change**: Downsample for overview, upsample for sub-pixel registration
3. **Custom DEM**: Producer used SRTM 30m, you have FABDEM 1m
4. **Co-registration**: Align with another dataset in a different coordinate system

### SIDD vs SICD for Ortho

| Aspect | SICD Ortho | SIDD Ortho |
|--------|-----------|------------|
| **Input** | Complex slant range | Real magnitude (pre-projected) |
| **Geolocation** | Nonlinear R/Rdot | Plane projection (affine-like) |
| **DEM use** | Native in R/Rdot inverse | Re-projection only (already projected) |
| **Accuracy** | Exact | Approximate (re-sampling existing projection) |
| **Speed** | Slower (iterative) | Faster (linear transform) |

**Recommendation**: Use SICD for highest fidelity. Use SIDD when speed matters or complex data unavailable.

## Summary

Successfully demonstrated:
- ✅ SIDD pre-projected product loading via IO factory
- ✅ SIDD geolocation auto-detection
- ✅ Re-orthorectification to custom UTM grid
- ✅ Chip extraction with offset handling
- ✅ Side-by-side original vs re-ortho visualization

### Key GRDL Patterns
- `get_reader('sidd', path)` for SIDD loading
- `create_geolocation(reader)` detects SIDD geolocation type
- Same `orthorectify()` API works for both SICD and SIDD
- `ChipGeolocation` wrapper for chip offsets

### Next Steps
- Compare SICD vs SIDD ortho for same scene (accuracy/speed trade-off)
- Try different output grids (ENUGrid, WebMercatorGrid)
- Export to GeoTIFF for GIS workflows
- Process full image (not just chip) with tiled orthorectification